In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes sentencepiece

In [ ]:
from transformers import LlamaTokenizer, LlamaForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset
from peft import get_peft_model, LoraConfig, TaskType
import json

In [ ]:
data_file = "dataset.jsonl" 

with open(data_file, "r") as f:
    data = [json.loads(line) for line in f]


dataset = Dataset.from_list(data)

In [ ]:
model_name = "codellama/CodeLlama-7b-Python-hf"
tokenizer = LlamaTokenizer.from_pretrained(model_name)

def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(tokenize, batched=True)

In [ ]:
model = LlamaForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="auto"
)

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir="./finetuned-codellama-manim",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    save_steps=500,
    logging_steps=20,
    save_total_limit=2
)

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("./finetuned-codellama-manim")
tokenizer.save_pretrained("./finetuned-codellama-manim")